## Fabric IQ Accelerator Sample
        
### Create Ontology from Package

In [1]:
# Install Fabric IQ Ontology Accelerator Package
%pip install /lakehouse/default/Files/fabriciq_ontology_accelerator-0.1.0-py3-none-any.whl --q
%pip install semantic-link-labs --q

StatementMeta(, 278a30b4-eb4f-4585-9331-276aea909ba0, 9, Finished, Available, Finished)


[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
fsspec-wrapper 0.1.15 requires PyJWT>=2.6.0, but you have pyjwt 2.4.0 which is incompatible.

[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.



In [5]:
import sempy.fabric as fabric
import json
import sempy_labs as labs
from fabricontology import create_ontology_item, generate_definition_from_package
from fabricontology.generate_data import generate_instance_data, generate_events_data
from notebookutils import mssparkutils

StatementMeta(, 278a30b4-eb4f-4585-9331-276aea909ba0, 14, Finished, Available, Finished)

In [8]:
lakehouse_name = "RetailLH"
eventhouse_name = "RetailEH"
workspace_id = fabric.get_workspace_id()
lakehouses = labs.lakehouse.list_lakehouses()
lakehouse_id = lakehouses[lakehouses["Lakehouse Name"] == lakehouse_name].reset_index()["Lakehouse ID"][0]
eventhouses = labs.list_eventhouses()
eventhouse_id = eventhouses[eventhouses["Eventhouse Name"] == eventhouse_name].reset_index()["Eventhouse Id"][0]
eventhouse_uri = eventhouses[eventhouses["Eventhouse Name"] == eventhouse_name].reset_index()["Query Service URI"][0]

StatementMeta(, 278a30b4-eb4f-4585-9331-276aea909ba0, 17, Finished, Available, Finished)

In [10]:
workspace_id = fabric.get_workspace_id()
access_token = notebookutils.credentials.getToken('pbi')

ontology_item_name = "RetailOntology"
ontology_package_path = "/lakehouse/default/Files/retail_ontology_package.iq"

binding_lakehouse_name = lakehouse_name
binding_lakehouse_schema_name = "dbo"  # replace this if using lakehouse without schemas
binding_eventhouse_name = eventhouse_name
binding_eventhouse_cluster_uri = eventhouse_uri
binding_eventhouse_database_name = eventhouse_name

items_df = fabric.list_items()
binding_lakehouse_item_id = str(items_df[(items_df["Type"] == "Lakehouse") & (items_df["Display Name"] == binding_lakehouse_name)].iloc[0].Id)
binding_eventhouse_item_id = str(items_df[(items_df["Type"] == "Eventhouse") & (items_df["Display Name"] == binding_eventhouse_name)].iloc[0].Id)
binding_workspace_id = workspace_id

ontology_definition, entity_types, relationship_types, data_bindings, contextualizations = generate_definition_from_package(
    ontology_package_path=ontology_package_path,
    ontology_name=ontology_item_name, 
    binding_workspace_id=binding_workspace_id,
    binding_lakehouse_item_id=binding_lakehouse_item_id,
    binding_lakehouse_schema_name=binding_lakehouse_schema_name,
    binding_eventhouse_item_id=binding_eventhouse_item_id,
    binding_eventhouse_cluster_uri=binding_eventhouse_cluster_uri,    
    binding_eventhouse_database_name=binding_eventhouse_database_name)

response = create_ontology_item(workspace_id=workspace_id, 
                           access_token=access_token,
                           ontology_item_name=ontology_item_name, 
                           ontology_definition=ontology_definition)
print(response.json())


StatementMeta(, 278a30b4-eb4f-4585-9331-276aea909ba0, 19, Submitted, Running, Running)

In [ ]:
ontology_package_path = "/lakehouse/default/Files/retail_ontology_package.iq"

# Create delta tables in the default lakehouse
lakehouse_schema = "dbo"  # replace this if using lakehouse without schemas.
response = generate_instance_data(spark, ontology_package_path=ontology_package_path, database=lakehouse_schema, mode="overwrite")

StatementMeta(, , -1, Waiting, , Waiting)

In [ ]:
# Create eventhouse tables 
eventhouse_cluster_uri = eventhouse_uri
eventhouse_database = eventhouse_name
access_token=mssparkutils.credentials.getToken(eventhouse_cluster_uri)

response = generate_events_data(spark, 
        ontology_package_path=ontology_package_path,
        eventhouse_cluster_uri=eventhouse_cluster_uri,
        eventhouse_database=eventhouse_database,
        access_token=access_token )

StatementMeta(, , -1, Waiting, , Waiting)